# OpenAI-compatible hello world

Minimal standalone notebook to verify a basic OpenAI-compatible chat-completions call.

It looks for `OPENAI_API_BASE_URL`, `OPENAI_API_KEY`, and `OPENAI_API_MODEL`, and will also try to read them from a local `.env` file.


In [ ]:
import os
from pathlib import Path

from openai import OpenAI


def load_local_env() -> None:
    for root in [Path.cwd(), *Path.cwd().parents]:
        env_path = root / ".env"
        if not env_path.exists():
            continue
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
        return


load_local_env()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_API_BASE_URL", "https://api.openai.com/v1")
model = os.getenv("OPENAI_API_MODEL", "gpt-5.4")

missing = [
    name
    for name, value in {
        "OPENAI_API_KEY": api_key,
        "OPENAI_API_MODEL": model,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Missing required env vars: {', '.join(missing)}")

client = OpenAI(api_key=api_key, base_url=base_url)
client


In [6]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Say hello world in one short sentence."},
    ],
    temperature=0,
)

print(response.choices[0].message.content)

Hello, world!
